In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 09_evaluate_test_and_holdout
# MAGIC Evaluar modelo en TEST y HOLDOUT (2018-10-01 a 2018-10-17)

# COMMAND ----------

import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pickle
from pyspark.sql import functions as F  # ####

SPLIT_PATH = "/Volumes/olist/olist_gold/model_split/"
GOLD_PATH = "/Volumes/olist/olist_gold/gold/"
METRICS_PATH = "/Volumes/olist/olist_gold/metrics/"
SILVER_PATH = "/Volumes/olist/olist_silver/silver/"

print("🚀 Iniciando evaluación TEST y HOLDOUT\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Cargar Mejor Modelo desde MLflow

# COMMAND ----------

# Reemplaza con el run_id del mejor modelo del Prompt 8
# O busca automáticamente el mejor por F1
BEST_RUN_ID = "510f08b94d9a436d9fa27731fa96e3fa"  # ⚠️ REEMPLAZAR con tu run_id

print(f"📦 Cargando modelo desde MLflow...\n")

model = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/model")

print(f"✅ Modelo cargado (run_id: {BEST_RUN_ID})\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Evaluación en TEST

# COMMAND ----------

print("="*60)
print("📊 EVALUACIÓN EN TEST")
print("="*60)

# Cargar test
test_df = spark.read.format("delta").load(f"{SPLIT_PATH}test/").toPandas()

X_test = test_df.drop(columns=["is_premium"])
y_test = test_df["is_premium"]

print(f"\n✅ Test: {len(test_df):,} registros\n")

# Predecir
y_pred_test = model.predict(X_test)

# Métricas
acc_test = accuracy_score(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)

print(f"📈 Métricas en TEST:")
print(f"  • Accuracy: {acc_test:.4f}")
print(f"  • F1-Score: {f1_test:.4f}\n")

print("📋 Classification Report:")
print(classification_report(y_test, y_pred_test))

print("\n📊 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_test))
print()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Preparar Datos HOLDOUT (2018-10-01 a 2018-10-17)

# COMMAND ----------

print("="*60)
print("📅 PREPARANDO HOLDOUT (2018-10-01 a 2018-10-17)")
print("="*60)

# Cargar orders_full original
orders_full = spark.read.format("delta").load(f"{SILVER_PATH}orders_full/")

# Filtrar periodo holdout
holdout_orders = orders_full.filter(
    (F.col("order_purchase_timestamp") >= "2018-10-01 00:00:00") &
    (F.col("order_purchase_timestamp") <= "2018-10-17 23:59:59") &
    (F.col("order_status") != "canceled")
).toPandas()

print(f"\n✅ Holdout: {len(holdout_orders):,} órdenes\n")

# COMMAND ----------

# Generar features para holdout (mismo proceso que Prompt 5)
print("🔧 Generando features para holdout...\n")

from pyspark.sql import functions as F

# Convertir de nuevo a Spark para agregaciones
holdout_spark = spark.createDataFrame(holdout_orders)

# Features agregadas por customer_id
holdout_features = holdout_spark.groupBy("customer_id").agg(
    F.datediff(F.lit("2018-10-17"), F.max("order_purchase_timestamp")).alias("recency"),
    F.count("order_id").alias("frequency"),
    F.sum("payment_sum").alias("monetary"),
    F.avg("payment_sum").alias("avg_ticket"),
    F.max("payment_sum").alias("max_ticket"),
    F.min("payment_sum").alias("min_ticket"),
    F.stddev("payment_sum").alias("std_ticket"),
    F.avg("items_count").alias("avg_items_per_order"),
    F.max("items_count").alias("max_items_per_order"),
    F.sum("items_count").alias("total_items"),
    F.avg("distinct_products").alias("avg_distinct_products"),
    F.sum("distinct_products").alias("total_distinct_products"),
    F.avg("sum_price").alias("avg_price"),
    F.sum("sum_price").alias("total_price"),
    F.avg("sum_freight").alias("avg_freight"),
    F.sum("sum_freight").alias("total_freight"),
    F.avg("avg_installments").alias("avg_installments"),
    F.max("avg_installments").alias("max_installments"),
    F.avg("n_payment_types").alias("avg_payment_types"),
    F.avg("avg_review_score").alias("avg_review_score"),
    F.min("avg_review_score").alias("min_review_score"),
    F.max("avg_review_score").alias("max_review_score"),
    F.count(F.when(F.col("avg_review_score").isNotNull(), 1)).alias("orders_with_review"),
    F.min("order_purchase_timestamp").alias("first_purchase"),
    F.max("order_purchase_timestamp").alias("last_purchase"),
    F.datediff(F.max("order_purchase_timestamp"), F.min("order_purchase_timestamp")).alias("customer_lifetime_days"),
    F.avg(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("avg_delivery_days"),
    F.max(F.datediff("order_delivered_customer_date", "order_purchase_timestamp")).alias("max_delivery_days"),
    F.avg(F.datediff("order_delivered_customer_date", "order_estimated_delivery_date")).alias("avg_delay_days"),
    F.count(F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)).alias("delayed_orders"),
    F.count(F.when(F.col("order_status") == "delivered", 1)).alias("delivered_orders"),
    F.count(F.when(F.col("order_status") == "shipped", 1)).alias("shipped_orders")
).toPandas()

# Features temporales
holdout_features['first_purchase_month'] = pd.to_datetime(holdout_features['first_purchase']).dt.month
holdout_features['first_purchase_day'] = pd.to_datetime(holdout_features['first_purchase']).dt.day
holdout_features['first_purchase_dow'] = pd.to_datetime(holdout_features['first_purchase']).dt.dayofweek
holdout_features['last_purchase_month'] = pd.to_datetime(holdout_features['last_purchase']).dt.month
holdout_features['last_purchase_day'] = pd.to_datetime(holdout_features['last_purchase']).dt.day
holdout_features['last_purchase_dow'] = pd.to_datetime(holdout_features['last_purchase']).dt.dayofweek

holdout_features = holdout_features.drop(columns=['first_purchase', 'last_purchase'])

# Features de interacción
holdout_features['freight_price_ratio'] = holdout_features['total_freight'] / holdout_features['total_price'].replace(0, 1)
holdout_features['monetary_per_order'] = holdout_features['monetary'] / holdout_features['frequency'].replace(0, 1)
holdout_features['items_per_monetary'] = holdout_features['total_items'] / holdout_features['monetary'].replace(0, 1)
holdout_features['products_per_order'] = holdout_features['total_distinct_products'] / holdout_features['frequency'].replace(0, 1)
holdout_features['review_score_x_monetary'] = holdout_features['avg_review_score'] * holdout_features['monetary']
holdout_features['delayed_ratio'] = holdout_features['delayed_orders'] / holdout_features['frequency'].replace(0, 1)
holdout_features['delivered_ratio'] = holdout_features['delivered_orders'] / holdout_features['frequency'].replace(0, 1)
holdout_features['orders_per_day'] = holdout_features['frequency'] / holdout_features['customer_lifetime_days'].replace(0, 1)

holdout_features = holdout_features.fillna(0)

print(f"✅ Features generadas: {len(holdout_features):,} clientes, {len(holdout_features.columns)} columnas\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Aplicar PCA al Holdout

# COMMAND ----------

print("🔬 Aplicando PCA al holdout...\n")

# Cargar customers_segmented para obtener labels reales de holdout (si existen)
customers_seg = spark.read.format("delta").load(f"{GOLD_PATH}customers_segmented_20180930/").toPandas()

# Merge para obtener is_premium real (si el cliente ya existía)
holdout_with_target = holdout_features.merge(
    customers_seg[['customer_id', 'is_premium']],
    on='customer_id',
    how='left'
)

# Clientes sin label (nuevos en holdout)
new_customers = holdout_with_target['is_premium'].isna().sum()
print(f"⚠️  {new_customers} clientes nuevos sin label histórico\n")

# Filtrar solo clientes con label para evaluación
holdout_labeled = holdout_with_target.dropna(subset=['is_premium'])

if len(holdout_labeled) == 0:
    print("❌ No hay clientes con label en holdout. No se puede evaluar.")
else:
    print(f"✅ {len(holdout_labeled):,} clientes con label para evaluar\n")
    
    # Preparar X
    X_holdout = holdout_labeled.drop(columns=['customer_id', 'is_premium'])
    y_holdout = holdout_labeled['is_premium'].astype(int)
    
    # Aplicar mismo pipeline: StandardScaler + PCA
    # Cargar el dataset reducido para obtener mismas columnas
    train_reduced = spark.read.format("delta").load(f"{GOLD_PATH}customer_features_rfm_20180930_reduced/").toPandas()
    pca_cols = [c for c in train_reduced.columns if c.startswith('pca_')]
    
    # Nota: Idealmente deberías guardar el scaler y PCA en Prompt 6
    # Como simplificación, re-entrenamos con los mismos datos
    print("⚠️  Re-entrenando PCA (idealmente cargar desde artifacts)\n")
    
    # Cargar features originales del train para PCA
    train_full = spark.read.format("delta").load(f"{GOLD_PATH}customer_features_rfm_20180930/").toPandas()
    X_train_full = train_full.drop(columns=['customer_id', 'is_premium', 'cluster_ordered']).fillna(0)
    
    # Estandarizar y PCA
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_full)
    
    pca = PCA(n_components=len(pca_cols))
    pca.fit(X_train_scaled)
    
    # Aplicar al holdout
    X_holdout_scaled = scaler.transform(X_holdout)
    X_holdout_pca = pca.transform(X_holdout_scaled)
    
    print(f"✅ Holdout transformado: {X_holdout_pca.shape}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Evaluación en HOLDOUT

# COMMAND ----------

if len(holdout_labeled) > 0:
    print("="*60)
    print("📊 EVALUACIÓN EN HOLDOUT")
    print("="*60)
    
    # Predecir
    y_pred_holdout = model.predict(X_holdout_pca)
    
    # Métricas
    acc_holdout = accuracy_score(y_holdout, y_pred_holdout)
    f1_holdout = f1_score(y_holdout, y_pred_holdout)
    
    print(f"\n📈 Métricas en HOLDOUT:")
    print(f"  • Accuracy: {acc_holdout:.4f}")
    print(f"  • F1-Score: {f1_holdout:.4f}\n")
    
    print("📋 Classification Report:")
    print(classification_report(y_holdout, y_pred_holdout))
    
    print("\n📊 Confusion Matrix:")
    print(confusion_matrix(y_holdout, y_pred_holdout))
    print()
else:
    acc_holdout = None
    f1_holdout = None

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Guardar Métricas

# COMMAND ----------

print("💾 Guardando métricas...\n")

metrics_data = [
    {"dataset": "test", "accuracy": acc_test, "f1_score": f1_test, "n_samples": len(y_test)},
]

if acc_holdout is not None:
    metrics_data.append({
        "dataset": "holdout", 
        "accuracy": acc_holdout, 
        "f1_score": f1_holdout, 
        "n_samples": len(y_holdout)
    })

metrics_df = pd.DataFrame(metrics_data)

spark.createDataFrame(metrics_df) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(f"{METRICS_PATH}model_metrics/")

print(f"✅ Guardado: {METRICS_PATH}model_metrics/\n")

# COMMAND ----------

# Resumen final
print(f"{'='*60}")
print("✅ EVALUACIÓN COMPLETADA")
print(f"{'='*60}")
print("\n📊 Resultados:")
print(f"\nTEST (split aleatorio 15%):")
print(f"  • Accuracy: {acc_test:.4f}")
print(f"  • F1-Score: {f1_test:.4f}")
print(f"  • Samples: {len(y_test):,}")

if acc_holdout is not None:
    print(f"\nHOLDOUT (2018-10-01 a 2018-10-17):")
    print(f"  • Accuracy: {acc_holdout:.4f}")
    print(f"  • F1-Score: {f1_holdout:.4f}")
    print(f"  • Samples: {len(y_holdout):,}")
else:
    print(f"\n⚠️  HOLDOUT: No evaluable (sin clientes con label histórico)")

print(f"\n💾 Métricas guardadas en: {METRICS_PATH}model_metrics/")